In [3]:
import cv2
import numpy as np
import torch

from pathlib import Path


ImageNet_Mean = np.array(
    [0.485, 0.456, 0.406],
    dtype=np.float32
)

ImageNet_Std = np.array(
    [0.229, 0.224, 0.225],
    dtype=np.float32
)


def Load_And_Preprocess_Video(
    Video_Path,
    Number_Of_Frames=16,
    Image_Size=224
):

    Video_Path = Path(
        Video_Path
    )

    if not Video_Path.exists():
        raise FileNotFoundError(
            f"Video not found: {Video_Path}"
        )

    Video_Capture = cv2.VideoCapture(
        str(Video_Path)
    )

    if not Video_Capture.isOpened():
        raise RuntimeError(
            f"Cannot open video: {Video_Path}"
        )

    Frame_Count = int(
        Video_Capture.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )

    if Frame_Count <= 0:
        Video_Capture.release()

        raise RuntimeError(
            f"Invalid frame count: {Video_Path}"
        )

    Frame_Indices = np.linspace(
        0,
        Frame_Count - 1,
        Number_Of_Frames,
        dtype=int
    )

    Processed_Frames = []

    for Frame_Index in Frame_Indices:

        Video_Capture.set(
            cv2.CAP_PROP_POS_FRAMES,
            int(Frame_Index)
        )

        Success, Frame = (
            Video_Capture.read()
        )

        if not Success:
            Video_Capture.release()

            raise RuntimeError(
                f"Cannot read frame "
                f"{Frame_Index} from "
                f"{Video_Path}"
            )

        Frame_RGB = cv2.cvtColor(
            Frame,
            cv2.COLOR_BGR2RGB
        )

        Gray_Frame = cv2.cvtColor(
            Frame_RGB,
            cv2.COLOR_RGB2GRAY
        )

        Resized_Gray_Frame = cv2.resize(
            Gray_Frame,
            (
                Image_Size,
                Image_Size
            ),
            interpolation=cv2.INTER_LINEAR
        )

        Frame_3Channel = np.repeat(
            Resized_Gray_Frame[
                :,
                :,
                np.newaxis
            ],
            3,
            axis=2
        )

        Processed_Frames.append(
            Frame_3Channel
        )

    Video_Capture.release()

    Processed_Frames = np.stack(
        Processed_Frames,
        axis=0
    )

    Normalized_Frames = (
        Processed_Frames.astype(
            np.float32
        )
        /
        255.0
    )

    Normalized_Frames = (
        Normalized_Frames
        -
        ImageNet_Mean
    ) / ImageNet_Std

    Video_Tensor = torch.from_numpy(
        Normalized_Frames
    )

    Video_Tensor = (
        Video_Tensor
        .permute(
            3,
            0,
            1,
            2
        )
        .contiguous()
    )

    Expected_Shape = (
        3,
        Number_Of_Frames,
        Image_Size,
        Image_Size
    )

    if Video_Tensor.shape != Expected_Shape:
        raise RuntimeError(
            f"Unexpected video shape: "
            f"{Video_Tensor.shape}"
        )

    if not torch.isfinite(
        Video_Tensor
    ).all():
        raise RuntimeError(
            f"NaN or Inf found in: "
            f"{Video_Path}"
        )

    return (
        Video_Tensor,
        Frame_Indices
    )


print(
    "Preprocessing function loaded successfully."
)

Preprocessing function loaded successfully.


In [4]:
import pandas as pd

Echonet_Dir = Path(
    "/kaggle/input/datasets/mahnurrahman/"
    "echonet-dynamic/EchoNet-Dynamic"
)

Videos_Dir = (
    Echonet_Dir
    /
    "Videos"
)

FileList_Path = (
    Echonet_Dir
    /
    "FileList.csv"
)

FileList = pd.read_csv(
    FileList_Path
)

Sample_Name = str(
    FileList.iloc[0]["FileName"]
)

Sample_Path = (
    Videos_Dir
    /
    f"{Sample_Name}.avi"
)

Video_Tensor, Frame_Indices = (
    Load_And_Preprocess_Video(
        Video_Path=Sample_Path,
        Number_Of_Frames=16,
        Image_Size=224
    )
)

print(
    "Shape:",
    Video_Tensor.shape
)

print(
    "Dtype:",
    Video_Tensor.dtype
)

print(
    "Finite:",
    torch.isfinite(
        Video_Tensor
    ).all().item()
)

print(
    "Frames:",
    Frame_Indices
)

Shape: torch.Size([3, 16, 224, 224])
Dtype: torch.float32
Finite: True
Frames: [  0  11  23  34  46  57  69  80  92 103 115 126 138 149 161 173]


In [ ]:
from transformers import AutoModel

MODEL_ID = "galilai-group/LeVJEPA-VideoMix-Large"

Device = torch.device("cuda:0")

Model = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    use_safetensors=True
)

Model = Model.to(Device)

Model.eval()

print("Model loaded successfully.")
print("Device:", next(Model.parameters()).device)

Total_Parameters = sum(
    Parameter.numel()
    for Parameter in Model.parameters()
)

print(
    "Parameters:",
    f"{Total_Parameters:,}"
)

In [ ]:
Input_Video = (
    Video_Tensor
    .unsqueeze(0)
    .to(Device)
)

print(
    "Input shape:",
    Input_Video.shape
)

with torch.inference_mode():

    Model_Output = Model(
        pixel_values=Input_Video
    )

print(
    "Output type:",
    type(Model_Output)
)

print(
    "Output keys:",
    Model_Output.keys()
)

print(
    "Last hidden state:",
    Model_Output.last_hidden_state.shape
)

print(
    "Pooler output:",
    Model_Output.pooler_output.shape
)

print(
    "Finite:",
    torch.isfinite(
        Model_Output.pooler_output
    ).all().item()
)

In [5]:
from torch.utils.data import Dataset, DataLoader

FileList = pd.read_csv(
    FileList_Path
)

Split_Values = (
    FileList["Split"]
    .astype(str)
    .str.upper()
)

Train_Data = (
    FileList[
        Split_Values == "TRAIN"
    ]
    .reset_index(drop=True)
)

Val_Data = (
    FileList[
        Split_Values == "VAL"
    ]
    .reset_index(drop=True)
)

Test_Data = (
    FileList[
        Split_Values == "TEST"
    ]
    .reset_index(drop=True)
)

EF_Mean = float(
    Train_Data["EF"].mean()
)

EF_Std = float(
    Train_Data["EF"].std()
)

print("TRAIN:", len(Train_Data))
print("VAL:", len(Val_Data))
print("TEST:", len(Test_Data))

print("EF Mean:", EF_Mean)
print("EF Std:", EF_Std)

TRAIN: 7465
VAL: 1288
TEST: 1277
EF Mean: 55.77764325112096
EF Std: 12.407189604685396


In [6]:
class EchoNetDataset(Dataset):

    def __init__(
        self,
        DataFrame,
        Videos_Dir,
        EF_Mean,
        EF_Std,
        Number_Of_Frames=16,
        Image_Size=224
    ):

        self.DataFrame = (
            DataFrame
            .reset_index(drop=True)
        )

        self.Videos_Dir = Path(
            Videos_Dir
        )

        self.EF_Mean = float(
            EF_Mean
        )

        self.EF_Std = float(
            EF_Std
        )

        self.Number_Of_Frames = (
            Number_Of_Frames
        )

        self.Image_Size = (
            Image_Size
        )


    def __len__(
        self
    ):

        return len(
            self.DataFrame
        )


    def __getitem__(
        self,
        Index
    ):

        Row = self.DataFrame.iloc[
            Index
        ]

        Video_Name = str(
            Row["FileName"]
        )

        EF = float(
            Row["EF"]
        )

        EF_Normalized = (
            EF - self.EF_Mean
        ) / self.EF_Std

        Video_Path = (
            self.Videos_Dir
            /
            f"{Video_Name}.avi"
        )

        Video_Tensor, Frame_Indices = (
            Load_And_Preprocess_Video(
                Video_Path=Video_Path,
                Number_Of_Frames=
                    self.Number_Of_Frames,
                Image_Size=
                    self.Image_Size
            )
        )

        return {
            "video":
                Video_Tensor,

            "ef":
                torch.tensor(
                    EF,
                    dtype=torch.float32
                ),

            "ef_normalized":
                torch.tensor(
                    EF_Normalized,
                    dtype=torch.float32
                ),

            "video_name":
                Video_Name,

            "frame_indices":
                torch.as_tensor(
                    Frame_Indices,
                    dtype=torch.long
                )
        }

In [ ]:
Train_Dataset = EchoNetDataset(
    Train_Data,
    Videos_Dir,
    EF_Mean,
    EF_Std
)

Val_Dataset = EchoNetDataset(
    Val_Data,
    Videos_Dir,
    EF_Mean,
    EF_Std
)

Test_Dataset = EchoNetDataset(
    Test_Data,
    Videos_Dir,
    EF_Mean,
    EF_Std
)

print("Train:", len(Train_Dataset))
print("Val:", len(Val_Dataset))
print("Test:", len(Test_Dataset))

In [ ]:
Sample = Train_Dataset[0]

print(
    Sample.keys()
)

print(
    "Video:",
    Sample["video"].shape
)

print(
    "EF:",
    Sample["ef"]
)

print(
    "EF normalized:",
    Sample["ef_normalized"]
)

print(
    "Video name:",
    Sample["video_name"]
)

In [ ]:
import gc

try:
    del Model
except:
    pass

try:
    del Model_Output
except:
    pass

try:
    del Input_Video
except:
    pass

gc.collect()
torch.cuda.empty_cache()

print(
    "GPU memory cleared."
)

In [7]:
import torch.nn as nn
from transformers import AutoModel


MODEL_ID = (
    "galilai-group/"
    "LeVJEPA-VideoMix-Large"
)


class LeVJEPA_EF_Model(nn.Module):

    def __init__(
        self,
        Model_ID
    ):

        super().__init__()

        self.Backbone = (
            AutoModel.from_pretrained(
                Model_ID,
                trust_remote_code=True,
                use_safetensors=True
            )
        )

        Hidden_Size = 1024

        self.EF_Head = nn.Linear(
            Hidden_Size,
            1
        )


    def forward(
        self,
        Pixel_Values
    ):

        Backbone_Output = (
            self.Backbone(
                pixel_values=
                    Pixel_Values
            )
        )

        Global_Feature = (
            Backbone_Output
            .pooler_output
        )

        EF_Prediction = (
            self.EF_Head(
                Global_Feature
            )
            .squeeze(-1)
        )

        return EF_Prediction


Model = LeVJEPA_EF_Model(
    MODEL_ID
)

config.json:   0%|          | 0.00/576 [00:00<?, ?B/s]

configuration_levjepa.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/galilai-group/LeVJEPA-VideoMix-Large:
- configuration_levjepa.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_levjepa.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/galilai-group/LeVJEPA-VideoMix-Large:
- modeling_levjepa.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/1.21G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

In [ ]:
if hasattr(
    Model.Backbone,
    "gradient_checkpointing_enable"
):

    try:

        Model.Backbone.gradient_checkpointing_enable()

        print(
            "Gradient checkpointing enabled."
        )

    except Exception as Error:

        print(
            "Gradient checkpointing unavailable:"
        )

        print(
            Error
        )

else:

    print(
        "Gradient checkpointing "
        "not supported."
    )

In [ ]:
Total_Parameters = sum(
    Parameter.numel()
    for Parameter in Model.parameters()
)

Trainable_Parameters = sum(
    Parameter.numel()
    for Parameter in Model.parameters()
    if Parameter.requires_grad
)

Frozen_Parameters = (
    Total_Parameters
    -
    Trainable_Parameters
)

print("Total:", f"{Total_Parameters:,}")
print("Trainable:", f"{Trainable_Parameters:,}")
print("Frozen:", f"{Frozen_Parameters:,}")

assert Frozen_Parameters == 0

print("\nFULL FINE-TUNING ENABLED.")

In [ ]:
Device = torch.device("cuda:0")

Model = Model.to(Device)

if torch.cuda.device_count() >= 2:

    Model = nn.DataParallel(
        Model,
        device_ids=[0, 1]
    )

    print("Multi-GPU enabled.")
    print("GPU 0:", torch.cuda.get_device_name(0))
    print("GPU 1:", torch.cuda.get_device_name(1))

else:

    print("Single GPU mode.")

In [ ]:
Batch_Size = 2

Gradient_Accumulation_Steps = 4

Effective_Batch_Size = (
    Batch_Size
    *
    Gradient_Accumulation_Steps
)

Num_Workers = 2


Train_Loader = DataLoader(
    Train_Dataset,
    batch_size=Batch_Size,
    shuffle=True,
    num_workers=Num_Workers,
    pin_memory=True,
    drop_last=False
)


Val_Loader = DataLoader(
    Val_Dataset,
    batch_size=Batch_Size,
    shuffle=False,
    num_workers=Num_Workers,
    pin_memory=True,
    drop_last=False
)


Test_Loader = DataLoader(
    Test_Dataset,
    batch_size=Batch_Size,
    shuffle=False,
    num_workers=Num_Workers,
    pin_memory=True,
    drop_last=False
)


print("Train batches:", len(Train_Loader))
print("Val batches:", len(Val_Loader))
print("Test batches:", len(Test_Loader))

print(
    "Effective batch size:",
    Effective_Batch_Size
)

In [ ]:
import math

from transformers import (
    get_cosine_schedule_with_warmup
)


Max_Epochs = 10

Early_Stopping_Patience = 3

Min_Improvement = 0.05


Backbone_Learning_Rate = 1e-5

Head_Learning_Rate = 1e-4

Weight_Decay = 0.01

Gradient_Clip_Norm = 1.0


Loss_Function = nn.MSELoss()


Base_Model = (
    Model.module
    if isinstance(Model, nn.DataParallel)
    else Model
)


Optimizer = torch.optim.AdamW(
    [
        {
            "params":
                Base_Model.Backbone.parameters(),

            "lr":
                Backbone_Learning_Rate
        },

        {
            "params":
                Base_Model.EF_Head.parameters(),

            "lr":
                Head_Learning_Rate
        }
    ],

    weight_decay=
        Weight_Decay
)


Optimizer_Steps_Per_Epoch = math.ceil(
    len(Train_Loader)
    /
    Gradient_Accumulation_Steps
)


Total_Optimizer_Steps = (
    Optimizer_Steps_Per_Epoch
    *
    Max_Epochs
)


Warmup_Steps = max(
    1,
    int(
        Total_Optimizer_Steps
        *
        0.05
    )
)


Scheduler = (
    get_cosine_schedule_with_warmup(
        Optimizer,
        num_warmup_steps=
            Warmup_Steps,
        num_training_steps=
            Total_Optimizer_Steps
    )
)


Scaler = torch.amp.GradScaler(
    "cuda"
)


print(
    "Optimizer steps / epoch:",
    Optimizer_Steps_Per_Epoch
)

print(
    "Maximum optimizer steps:",
    Total_Optimizer_Steps
)

print(
    "Warmup steps:",
    Warmup_Steps
)

print(
    "AMP: FP16"
)

print(
    "\nFULL FINE-TUNING SETUP READY."
)

In [ ]:
import os
import time
import random
import numpy as np
import pandas as pd

from pathlib import Path
from torch.utils.data import Subset


Output_Dir = Path(
    "/kaggle/working/echonet_levjepa_output"
)

Checkpoints_Dir = (
    Output_Dir / "checkpoints"
)

Logs_Dir = (
    Output_Dir / "logs"
)

Checkpoints_Dir.mkdir(
    parents=True,
    exist_ok=True
)

Logs_Dir.mkdir(
    parents=True,
    exist_ok=True
)


Last_Checkpoint_Path = (
    Checkpoints_Dir /
    "full_finetuning_last.pt"
)

Best_Model_Path = (
    Checkpoints_Dir /
    "full_finetuning_best.pt"
)

History_Path = (
    Logs_Dir /
    "training_history.csv"
)


Checkpoint_Every = 250
Print_Every = 10


# ------------------------------------------------
# Deterministic train order
# ------------------------------------------------

Seed = 42

Batches_Per_Epoch = math.ceil(
    len(Train_Dataset) /
    Batch_Size
)


def Create_Train_Loader(
    Epoch,
    Start_Batch_Index=0
):

    Generator = torch.Generator()

    Generator.manual_seed(
        Seed + Epoch
    )

    Epoch_Indices = (
        torch.randperm(
            len(Train_Dataset),
            generator=Generator
        )
        .tolist()
    )

    Start_Sample_Index = (
        Start_Batch_Index *
        Batch_Size
    )

    Remaining_Indices = (
        Epoch_Indices[
            Start_Sample_Index:
        ]
    )

    Epoch_Dataset = Subset(
        Train_Dataset,
        Remaining_Indices
    )

    return DataLoader(
        Epoch_Dataset,
        batch_size=Batch_Size,
        shuffle=False,
        num_workers=Num_Workers,
        pin_memory=True,
        drop_last=False
    )


# ------------------------------------------------
# Validation
# ------------------------------------------------

def Validate_Model():

    Model.eval()

    Total_Loss = 0.0
    Total_Samples = 0

    Predictions = []
    Targets = []

    with torch.inference_mode():

        for Batch in Val_Loader:

            Videos = (
                Batch["video"]
                .to(
                    Device,
                    non_blocking=True
                )
            )

            Targets_Normalized = (
                Batch["ef_normalized"]
                .to(
                    Device,
                    non_blocking=True
                )
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                Predictions_Normalized = (
                    Model(Videos)
                )

                Loss = Loss_Function(
                    Predictions_Normalized,
                    Targets_Normalized
                )

            Current_Batch_Size = (
                Videos.shape[0]
            )

            Total_Loss += (
                Loss.item() *
                Current_Batch_Size
            )

            Total_Samples += (
                Current_Batch_Size
            )

            Predictions_Real = (
                Predictions_Normalized
                .float()
                *
                EF_Std
                +
                EF_Mean
            )

            Predictions.extend(
                Predictions_Real
                .cpu()
                .numpy()
                .tolist()
            )

            Targets.extend(
                Batch["ef"]
                .numpy()
                .tolist()
            )


    Predictions = np.asarray(
        Predictions,
        dtype=np.float32
    )

    Targets = np.asarray(
        Targets,
        dtype=np.float32
    )


    Val_Loss = (
        Total_Loss /
        Total_Samples
    )

    Val_MAE = float(
        np.mean(
            np.abs(
                Predictions -
                Targets
            )
        )
    )

    Val_RMSE = float(
        np.sqrt(
            np.mean(
                (
                    Predictions -
                    Targets
                ) ** 2
            )
        )
    )

    Model.train()

    return (
        Val_Loss,
        Val_MAE,
        Val_RMSE
    )


# ------------------------------------------------
# Checkpoint
# ------------------------------------------------

def Save_Last_Checkpoint(
    Epoch,
    Batch_Index,
    Global_Step,
    Best_Val_MAE,
    Early_Stopping_Best_MAE,
    Epochs_Without_Improvement,
    Epoch_Loss_Sum,
    Epoch_Sample_Count,
    Training_History,
    Epoch_Complete
):

    Checkpoint = {

        "epoch":
            int(Epoch),

        "batch_index":
            int(Batch_Index),

        "epoch_complete":
            bool(Epoch_Complete),

        "global_step":
            int(Global_Step),

        "best_val_mae":
            float(Best_Val_MAE),

        "early_stopping_best_mae":
            float(
                Early_Stopping_Best_MAE
            ),

        "epochs_without_improvement":
            int(
                Epochs_Without_Improvement
            ),

        "epoch_loss_sum":
            float(Epoch_Loss_Sum),

        "epoch_sample_count":
            int(Epoch_Sample_Count),

        "training_history":
            Training_History,

        "ef_mean":
            EF_Mean,

        "ef_std":
            EF_Std,

        "model_state_dict":
            Base_Model.state_dict(),

        "optimizer_state_dict":
            Optimizer.state_dict(),

        "scheduler_state_dict":
            Scheduler.state_dict(),

        "scaler_state_dict":
            Scaler.state_dict()
    }


    Temp_Path = Path(
        str(Last_Checkpoint_Path)
        + ".tmp"
    )

    torch.save(
        Checkpoint,
        Temp_Path
    )

    os.replace(
        Temp_Path,
        Last_Checkpoint_Path
    )


def Save_Best_Model(
    Epoch,
    Val_Loss,
    Val_MAE,
    Val_RMSE
):

    Data = {

        "epoch":
            int(Epoch),

        "ef_mean":
            EF_Mean,

        "ef_std":
            EF_Std,

        "val_loss":
            float(Val_Loss),

        "val_mae":
            float(Val_MAE),

        "val_rmse":
            float(Val_RMSE),

        "model_state_dict":
            Base_Model.state_dict()
    }


    Temp_Path = Path(
        str(Best_Model_Path)
        + ".tmp"
    )

    torch.save(
        Data,
        Temp_Path
    )

    os.replace(
        Temp_Path,
        Best_Model_Path
    )


print("Training helpers ready.")
print("Output:", Output_Dir)

In [ ]:
Start_Epoch = 0
Start_Batch_Index = 0

Global_Step = 0

Best_Val_MAE = float("inf")

Early_Stopping_Best_MAE = float(
    "inf"
)

Epochs_Without_Improvement = 0

Training_History = []

Resume_Epoch_Loss_Sum = 0.0
Resume_Epoch_Sample_Count = 0


if Last_Checkpoint_Path.exists():

    print(
        "Checkpoint found."
    )

    Checkpoint = torch.load(
        Last_Checkpoint_Path,
        map_location="cpu",
        weights_only=False
    )


    Base_Model.load_state_dict(
        Checkpoint[
            "model_state_dict"
        ]
    )

    Optimizer.load_state_dict(
        Checkpoint[
            "optimizer_state_dict"
        ]
    )

    Scheduler.load_state_dict(
        Checkpoint[
            "scheduler_state_dict"
        ]
    )

    Scaler.load_state_dict(
        Checkpoint[
            "scaler_state_dict"
        ]
    )


    Base_Model.to(
        Device
    )


    Global_Step = int(
        Checkpoint[
            "global_step"
        ]
    )

    Best_Val_MAE = float(
        Checkpoint[
            "best_val_mae"
        ]
    )

    Early_Stopping_Best_MAE = float(
        Checkpoint.get(
            "early_stopping_best_mae",
            Best_Val_MAE
        )
    )

    Epochs_Without_Improvement = int(
        Checkpoint.get(
            "epochs_without_improvement",
            0
        )
    )

    Training_History = (
        Checkpoint.get(
            "training_history",
            []
        )
    )


    Saved_Epoch = int(
        Checkpoint["epoch"]
    )

    Saved_Batch = int(
        Checkpoint["batch_index"]
    )


    if Checkpoint.get(
        "epoch_complete",
        False
    ):

        Start_Epoch = (
            Saved_Epoch + 1
        )

        Start_Batch_Index = 0

    else:

        Start_Epoch = (
            Saved_Epoch
        )

        Start_Batch_Index = (
            Saved_Batch + 1
        )

        Resume_Epoch_Loss_Sum = float(
            Checkpoint.get(
                "epoch_loss_sum",
                0.0
            )
        )

        Resume_Epoch_Sample_Count = int(
            Checkpoint.get(
                "epoch_sample_count",
                0
            )
        )


    print(
        "Resume epoch:",
        Start_Epoch + 1
    )

    print(
        "Resume batch:",
        Start_Batch_Index + 1
    )

    print(
        "Global step:",
        Global_Step
    )

    print(
        "Best Val MAE:",
        Best_Val_MAE
    )


else:

    print(
        "No checkpoint found."
    )

    print(
        "Starting new full fine-tuning run."
    )

In [ ]:
Model.train()

Optimizer.zero_grad(
    set_to_none=True
)

Training_Start_Time = (
    time.time()
)


for Epoch in range(
    Start_Epoch,
    Max_Epochs
):

    Epoch_Start_Time = (
        time.time()
    )


    if Epoch == Start_Epoch:

        Epoch_Start_Batch = (
            Start_Batch_Index
        )

        Epoch_Loss_Sum = (
            Resume_Epoch_Loss_Sum
        )

        Epoch_Sample_Count = (
            Resume_Epoch_Sample_Count
        )

    else:

        Epoch_Start_Batch = 0

        Epoch_Loss_Sum = 0.0

        Epoch_Sample_Count = 0


    Train_Loader_Current = (
        Create_Train_Loader(
            Epoch,
            Epoch_Start_Batch
        )
    )


    print(
        "\n"
        + "=" * 70
    )

    print(
        f"EPOCH "
        f"{Epoch + 1}/"
        f"{Max_Epochs}"
    )

    print(
        "=" * 70
    )


    for Local_Batch_Index, Batch in enumerate(
        Train_Loader_Current
    ):

        Batch_Index = (
            Epoch_Start_Batch
            +
            Local_Batch_Index
        )


        Videos = (
            Batch["video"]
            .to(
                Device,
                non_blocking=True
            )
        )


        Targets_Normalized = (
            Batch[
                "ef_normalized"
            ]
            .to(
                Device,
                non_blocking=True
            )
        )


        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            Predictions_Normalized = (
                Model(
                    Videos
                )
            )

            Raw_Loss = (
                Loss_Function(
                    Predictions_Normalized,
                    Targets_Normalized
                )
            )


        if not torch.isfinite(
            Raw_Loss
        ):

            raise RuntimeError(
                f"Non-finite loss at "
                f"Epoch {Epoch + 1}, "
                f"Batch {Batch_Index + 1}"
            )


        Group_Start = (
            Batch_Index
            //
            Gradient_Accumulation_Steps
        ) * Gradient_Accumulation_Steps


        Current_Accumulation_Size = min(
            Gradient_Accumulation_Steps,
            Batches_Per_Epoch
            -
            Group_Start
        )


        Loss = (
            Raw_Loss
            /
            Current_Accumulation_Size
        )


        Scaler.scale(
            Loss
        ).backward()


        Current_Batch_Size = (
            Videos.shape[0]
        )


        Epoch_Loss_Sum += (
            Raw_Loss.item()
            *
            Current_Batch_Size
        )

        Epoch_Sample_Count += (
            Current_Batch_Size
        )


        Should_Step = (
            (
                Batch_Index + 1
            )
            %
            Gradient_Accumulation_Steps
            ==
            0
        )


        Is_Last_Batch = (
            Batch_Index + 1
            ==
            Batches_Per_Epoch
        )


        if (
            Should_Step
            or
            Is_Last_Batch
        ):

            Scaler.unscale_(
                Optimizer
            )


            Gradient_Norm = (
                torch.nn.utils
                .clip_grad_norm_(
                    Base_Model.parameters(),
                    Gradient_Clip_Norm
                )
            )


            Scaler.step(
                Optimizer
            )

            Scaler.update()

            Scheduler.step()


            Optimizer.zero_grad(
                set_to_none=True
            )


            Global_Step += 1


            if (
                Global_Step
                %
                Print_Every
                ==
                0
                or
                Global_Step == 1
            ):

                Prediction_Real = (
                    Predictions_Normalized[
                        0
                    ]
                    .detach()
                    .float()
                    .item()
                    *
                    EF_Std
                    +
                    EF_Mean
                )


                Target_Real = (
                    Batch["ef"][
                        0
                    ].item()
                )


                GPU0_Memory = (
                    torch.cuda
                    .memory_allocated(0)
                    /
                    1024**3
                )

                GPU1_Memory = (
                    torch.cuda
                    .memory_allocated(1)
                    /
                    1024**3
                )


                Elapsed = (
                    time.time()
                    -
                    Training_Start_Time
                )


                print(
                    f"Epoch "
                    f"{Epoch + 1}/"
                    f"{Max_Epochs} | "
                    f"Batch "
                    f"{Batch_Index + 1}/"
                    f"{Batches_Per_Epoch} | "
                    f"Step "
                    f"{Global_Step} | "
                    f"Loss "
                    f"{Raw_Loss.item():.4f} | "
                    f"Pred EF "
                    f"{Prediction_Real:.2f} | "
                    f"Target EF "
                    f"{Target_Real:.2f} | "
                    f"Grad "
                    f"{float(Gradient_Norm):.3f} | "
                    f"GPU0 "
                    f"{GPU0_Memory:.2f}GB | "
                    f"GPU1 "
                    f"{GPU1_Memory:.2f}GB | "
                    f"Time "
                    f"{Elapsed / 3600:.2f}h"
                )


            if (
                Global_Step
                %
                Checkpoint_Every
                ==
                0
            ):

                Save_Last_Checkpoint(
                    Epoch=
                        Epoch,

                    Batch_Index=
                        Batch_Index,

                    Global_Step=
                        Global_Step,

                    Best_Val_MAE=
                        Best_Val_MAE,

                    Early_Stopping_Best_MAE=
                        Early_Stopping_Best_MAE,

                    Epochs_Without_Improvement=
                        Epochs_Without_Improvement,

                    Epoch_Loss_Sum=
                        Epoch_Loss_Sum,

                    Epoch_Sample_Count=
                        Epoch_Sample_Count,

                    Training_History=
                        Training_History,

                    Epoch_Complete=
                        False
                )

                print(
                    "\nCheckpoint saved."
                )


    Train_Loss = (
        Epoch_Loss_Sum
        /
        Epoch_Sample_Count
    )


    print(
        f"\nTrain Loss: "
        f"{Train_Loss:.6f}"
    )


    print(
        "Running validation..."
    )


    Val_Loss, Val_MAE, Val_RMSE = (
        Validate_Model()
    )


    print(
        f"Val Loss: "
        f"{Val_Loss:.6f}"
    )

    print(
        f"Val MAE: "
        f"{Val_MAE:.4f}"
    )

    print(
        f"Val RMSE: "
        f"{Val_RMSE:.4f}"
    )


    # Best actual model
    if Val_MAE < Best_Val_MAE:

        Best_Val_MAE = float(
            Val_MAE
        )

        Save_Best_Model(
            Epoch,
            Val_Loss,
            Val_MAE,
            Val_RMSE
        )

        print(
            "NEW BEST MODEL SAVED."
        )


    # Early stopping criterion
    Significant_Improvement = (
        Val_MAE
        <
        Early_Stopping_Best_MAE
        -
        Min_Improvement
    )


    if Significant_Improvement:

        Early_Stopping_Best_MAE = (
            float(
                Val_MAE
            )
        )

        Epochs_Without_Improvement = 0

    else:

        Epochs_Without_Improvement += 1


    Training_History.append(
        {
            "epoch":
                Epoch + 1,

            "train_loss":
                float(
                    Train_Loss
                ),

            "val_loss":
                float(
                    Val_Loss
                ),

            "val_mae":
                float(
                    Val_MAE
                ),

            "val_rmse":
                float(
                    Val_RMSE
                ),

            "best_val_mae":
                float(
                    Best_Val_MAE
                ),

            "early_stopping_counter":
                int(
                    Epochs_Without_Improvement
                ),

            "global_step":
                int(
                    Global_Step
                )
        }
    )


    pd.DataFrame(
        Training_History
    ).to_csv(
        History_Path,
        index=False
    )


    Save_Last_Checkpoint(
        Epoch=
            Epoch,

        Batch_Index=
            Batches_Per_Epoch - 1,

        Global_Step=
            Global_Step,

        Best_Val_MAE=
            Best_Val_MAE,

        Early_Stopping_Best_MAE=
            Early_Stopping_Best_MAE,

        Epochs_Without_Improvement=
            Epochs_Without_Improvement,

        Epoch_Loss_Sum=
            Epoch_Loss_Sum,

        Epoch_Sample_Count=
            Epoch_Sample_Count,

        Training_History=
            Training_History,

        Epoch_Complete=
            True
    )


    Epoch_Time = (
        time.time()
        -
        Epoch_Start_Time
    )


    print(
        f"Epoch time: "
        f"{Epoch_Time / 3600:.2f}h"
    )

    print(
        f"Best Val MAE: "
        f"{Best_Val_MAE:.4f}"
    )

    print(
        "Early stopping:",
        f"{Epochs_Without_Improvement}/"
        f"{Early_Stopping_Patience}"
    )


    if (
        Epochs_Without_Improvement
        >=
        Early_Stopping_Patience
    ):

        print(
            "\nEARLY STOPPING."
        )

        break


print(
    "\nFULL FINE-TUNING FINISHED."
)

print(
    "Best model:",
    Best_Model_Path
)

print(
    "Last checkpoint:",
    Last_Checkpoint_Path
)

In [8]:
Best_Model_Path = Path(
    "/kaggle/input/notebooks/mahedrashidi/echonet/"
    "echonet_levjepa_output/checkpoints/"
    "full_finetuning_best.pt"
)

print(
    "Checkpoint exists:",
    Best_Model_Path.exists()
)


Best_Model_Data = torch.load(
    Best_Model_Path,
    map_location="cpu",
    weights_only=False
)


print(
    "Best Epoch:",
    Best_Model_Data["epoch"] + 1
)

print(
    "Best Val MAE:",
    Best_Model_Data["val_mae"]
)

print(
    "Best Val RMSE:",
    Best_Model_Data["val_rmse"]
)


# حتماً normalization دقیق خود training را استفاده کن
EF_Mean = float(
    Best_Model_Data["ef_mean"]
)

EF_Std = float(
    Best_Model_Data["ef_std"]
)


Model.load_state_dict(
    Best_Model_Data[
        "model_state_dict"
    ]
)


print(
    "Fine-tuned weights loaded successfully."
)

Checkpoint exists: True
Best Epoch: 5
Best Val MAE: 4.405430793762207
Best Val RMSE: 5.911117076873779
Fine-tuned weights loaded successfully.


In [9]:
Device = torch.device(
    "cuda:0"
)

Model = Model.to(
    Device
)


if torch.cuda.device_count() >= 2:

    Model = nn.DataParallel(
        Model,
        device_ids=[0, 1]
    )

    print(
        "Using 2 GPUs."
    )

else:

    print(
        "Using 1 GPU."
    )


Model.eval()

Using 2 GPUs.


DataParallel(
  (module): LeVJEPA_EF_Model(
    (Backbone): LeVJEPAModel(
      (encoder): VisionTransformer(
        (patch_embed): PatchEmbed3D(
          (proj): Conv3d(3, 1024, kernel_size=(1, 16, 16), stride=(1, 16, 16))
        )
        (blocks): ModuleList(
          (0-23): 24 x Block(
            (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
            (attn): RoPEAttention(
              (qkv): Linear(in_features=1024, out_features=3072, bias=True)
              (attn_drop): Dropout(p=0.0, inplace=False)
              (proj): Linear(in_features=1024, out_features=1024, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
            )
            (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
            (mlp): MLP(
              (fc1): Linear(in_features=1024, out_features=4096, bias=True)
              (act): GELU(approximate='none')
              (fc2): Linear(in_features=4096, out_features=1024, bias=True)
              

In [11]:
Test_Dataset = EchoNetDataset(
    Test_Data,
    Videos_Dir,
    EF_Mean,
    EF_Std,
    Number_Of_Frames=16,
    Image_Size=224
)

print(
    "Test Dataset ready:",
    len(Test_Dataset)
)

Test Dataset ready: 1277


In [12]:
Test_Loader = DataLoader(
    Test_Dataset,
    batch_size=2,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    drop_last=False
)

print(
    "Test batches:",
    len(Test_Loader)
)

Test batches: 639


In [13]:
Loss_Function = nn.MSELoss()


Test_Predictions = []
Test_Targets = []
Test_Video_Names = []

Test_Loss_Sum = 0.0
Test_Sample_Count = 0


Model.eval()


with torch.inference_mode():

    for Batch in Test_Loader:

        Videos = (
            Batch["video"]
            .to(
                Device,
                non_blocking=True
            )
        )

        Targets_Normalized = (
            Batch["ef_normalized"]
            .to(
                Device,
                non_blocking=True
            )
        )


        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            Predictions_Normalized = (
                Model(Videos)
            )

            Loss = (
                Loss_Function(
                    Predictions_Normalized,
                    Targets_Normalized
                )
            )


        Current_Batch_Size = (
            Videos.shape[0]
        )

        Test_Loss_Sum += (
            Loss.item()
            *
            Current_Batch_Size
        )

        Test_Sample_Count += (
            Current_Batch_Size
        )


        Predictions_Real = (
            Predictions_Normalized
            .float()
            *
            EF_Std
            +
            EF_Mean
        )


        Test_Predictions.extend(
            Predictions_Real
            .cpu()
            .numpy()
            .tolist()
        )


        Test_Targets.extend(
            Batch["ef"]
            .numpy()
            .tolist()
        )


        Test_Video_Names.extend(
            Batch["video_name"]
        )


Test_Predictions = np.asarray(
    Test_Predictions,
    dtype=np.float32
)

Test_Targets = np.asarray(
    Test_Targets,
    dtype=np.float32
)


Test_Loss = (
    Test_Loss_Sum
    /
    Test_Sample_Count
)


Test_MAE = float(
    np.mean(
        np.abs(
            Test_Predictions
            -
            Test_Targets
        )
    )
)


Test_RMSE = float(
    np.sqrt(
        np.mean(
            (
                Test_Predictions
                -
                Test_Targets
            ) ** 2
        )
    )
)


SS_Residual = np.sum(
    (
        Test_Targets
        -
        Test_Predictions
    ) ** 2
)

SS_Total = np.sum(
    (
        Test_Targets
        -
        Test_Targets.mean()
    ) ** 2
)


Test_R2 = float(
    1.0
    -
    SS_Residual
    /
    SS_Total
)


print()
print("FINAL TEST RESULTS")
print("==================")

print(
    "Test Loss:",
    Test_Loss
)

print(
    "Test MAE:",
    Test_MAE
)

print(
    "Test RMSE:",
    Test_RMSE
)

print(
    "Test R²:",
    Test_R2
)

/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)



FINAL TEST RESULTS
Test Loss: 0.24035918447085108
Test MAE: 4.519306182861328
Test RMSE: 6.082803726196289
Test R²: 0.7524431943893433


In [14]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

Results_Dir = Path(
    "/kaggle/working/echonet_levjepa_output/results"
)

Results_Dir.mkdir(
    parents=True,
    exist_ok=True
)

Test_Results = pd.DataFrame({
    "video_name": Test_Video_Names,
    "ef_target": Test_Targets,
    "ef_prediction": Test_Predictions,
    "absolute_error": np.abs(
        Test_Targets - Test_Predictions
    )
})

Test_Results.to_csv(
    Results_Dir / "test_predictions.csv",
    index=False
)

Metrics = {
    "test_loss": float(Test_Loss),
    "test_mae": float(Test_MAE),
    "test_rmse": float(Test_RMSE),
    "test_r2": float(Test_R2)
}

with open(
    Results_Dir / "test_metrics.json",
    "w"
) as File:
    json.dump(
        Metrics,
        File,
        indent=4
    )

print("Saved:")
print(
    Results_Dir /
    "test_predictions.csv"
)
print(
    Results_Dir /
    "test_metrics.json"
)

Saved:
/kaggle/working/echonet_levjepa_output/results/test_predictions.csv
/kaggle/working/echonet_levjepa_output/results/test_metrics.json
